<a href="https://colab.research.google.com/github/muhammadghazali-bit/ProgramDS/blob/main/Tugas_Regresi__Ghaza.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **VISUALISASI REGRESI**

# **WEB APP REGRESX**
Kalkulator n Visualisasi Regresi


**klik [RegresX](https://script.google.com/macros/s/AKfycbyKj6crJ_1EXc2QGqeM9WBl2gIi20gUhZFuJrgnHiOhYQvOWJ1ooLgOfQhItVhBimrO/exec)**

# **Analisi pengaruh belanja modal fiskal terhadap tingkat pengangguran di Kota Banda Aceh.**


**Variabel yang Diukur:**

Variabel Terikat (Y): PENGANGGURAN (Tingkat Pengangguran).

Variabel Bebas (X1) : bm_ekonomi (Belanja Modal Fungsi Ekonomi).

Variabel Bebas (X2): bm_perlindungan_sosial (Belanja Modal Fungsi Perlindungan Sosial).

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.stats.api as sms
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
import plotly.graph_objects as go
import plotly.io as pio


pio.renderers.default = 'colab'

# -----------------------------------------------------------------------------------------
# 1. MEMBACA DATA ASLI

url = 'https://raw.githubusercontent.com/Alam-A99/DATAAKSI/main/dataset_fiskal_97_hipoFull.csv'
df_all = pd.read_csv(url)
df = df_all[df_all['Kota'] == 'Kota Banda Aceh'].copy()

print(f"Total data Banda Aceh yang dianalisis: {len(df)} baris.\n")

# -----------------------------------------------------------------------------------------
# 2. MODEL REGRESI & UJI ASUMSI KLASIK

X = df[['bm_ekonomi', 'bm_perlindungan_sosial']]
Y = df['PENGANGGURAN']
X_sm = sm.add_constant(X)
model = sm.OLS(Y, X_sm).fit()

# Menampilkan Summary OLS
print(model.summary())

print("\n" + "="*50)
print("FUNGSI REGRESI:")
print(f"Y = {model.params['const']:.4f} + {model.params['bm_ekonomi']:.4f}(BM_Ekonomi) - {abs(model.params['bm_perlindungan_sosial']):.4f}(BM_Sosial)")
print("="*50)

print("\nHASIL UJI ASUMSI KLASIK")
# 1. Normalitas
shapiro_stat, shapiro_p = stats.shapiro(model.resid)
print(f"- Normalitas (Shapiro-Wilk) P-Value: {shapiro_p:.4f} (>0.05 = Lolos)")

# 2. Multikolinearitas (VIF)
vif_data = pd.DataFrame()
vif_data["Variabel"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(f"- Multikolinearitas (VIF)            : {vif_data['VIF'].max():.4f} (<10 = Lolos)")

# 3. Heteroskedastisitas
bp_test = sms.het_breuschpagan(model.resid, model.model.exog)
print(f"- Heteroskedastisitas (BP) P-Value   : {bp_test[1]:.4f} (>0.05 = Lolos)")

# 4. Autokorelasi
dw_value = sm.stats.stattools.durbin_watson(model.resid)
print(f"- Autokorelasi (Durbin-Watson)       : {dw_value:.4f} (Mendekati 2 = Lolos)\n")

# --------------------------------------------------------------------------------
# 3. VISUALISASI 3D INTERAKTIF


# Membuat grid bidang regresi berdasarkan rentang data asli
x_range = np.linspace(df['bm_ekonomi'].min(), df['bm_ekonomi'].max(), 20)
y_range = np.linspace(df['bm_perlindungan_sosial'].min(), df['bm_perlindungan_sosial'].max(), 20)
x_grid, y_grid = np.meshgrid(x_range, y_range)

# Menghitung prediksi Z untuk permukaan bidang
z_grid = model.params['const'] + (model.params['bm_ekonomi'] * x_grid) + (model.params['bm_perlindungan_sosial'] * y_grid)

# Inisialisasi Plotly Figure
fig = go.Figure()

# Menambahkan Bidang Regresi Permukaan
fig.add_trace(go.Surface(
    x=x_grid,
    y=y_grid,
    z=z_grid,
    name='Bidang Regresi',
    colorscale='viridis',
    opacity=0.8,
    showscale=False
))

# Menambahkan Titik Data Aktual Asli
fig.add_trace(go.Scatter3d(
    x=df['bm_ekonomi'],
    y=df['bm_perlindungan_sosial'],
    z=df['PENGANGGURAN'],
    mode='markers',
    name='Data Aktual',
    marker=dict(size=6, color='red', line=dict(color='white', width=1))
))

# Mengatur Tampilan Layout Grafik
fig.update_layout(
    title=f"Plot Bidang Regresi 3D: Pengangguran terhadap BM Ekonomi & BM Sosial (Banda Aceh) | R² = {model.rsquared:.2f}",
    template='plotly_dark',
    scene=dict(
        xaxis_title='BM Ekonomi',
        yaxis_title='BM Perlindungan Sosial',
        zaxis_title='Pengangguran'
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

# Menampilkan kanvas interaktif di bawah sel Colab
fig.show()

Total data Banda Aceh yang dianalisis: 11 baris.

                            OLS Regression Results                            
Dep. Variable:           PENGANGGURAN   R-squared:                       0.271
Model:                            OLS   Adj. R-squared:                  0.089
Method:                 Least Squares   F-statistic:                     1.486
Date:                Thu, 15 Oct 2026   Prob (F-statistic):              0.283
Time:                        01:42:34   Log-Likelihood:                -22.988
No. Observations:                  11   AIC:                             51.98
Df Residuals:                       8   BIC:                             53.17
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------

# **Menguji pengaruh satu variabel bebas secara spesifik terhadap tingkat pengangguran.**

**Variabel yang Diukur**

Variabel Bebas / Independen (X): bm_ekonomi (Belanja Modal Fungsi Ekonomi)   

Variabel Terikat / Dependen (Y): PENGANGGURAN (Tingkat Pengangguran)

In [ ]:
# -----------------------------------------------------------------------------------------
# REGRESI LINEAR 2 VARIABEL (BM EKONOMI vs PENGANGGURAN)
# -----------------------------------------------------------------------------------------


import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.stats.api as sms
from scipy import stats
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = 'colab'

# 1. Menentukan Variabel (1 Independen X, 1 Dependen Y)
X_2v = df[['bm_ekonomi']]
Y_2v = df['PENGANGGURAN']

# Menambahkan konstanta dan melatih model
X_2v_sm = sm.add_constant(X_2v)
model_2v = sm.OLS(Y_2v, X_2v_sm).fit()

# 2. Output Ringkasan Model & Persamaan Regresi
print("="*50)
print("HASIL REGRESI 2 VARIABEL")
print("="*50)
print(model_2v.summary())

print("\n" + "="*50)
print("FUNGSI REGRESI 2 VARIABEL:")
print(f"Y = {model_2v.params['const']:.4f} + {model_2v.params['bm_ekonomi']:.4f}(BM_Ekonomi)")
print("="*50)

# 3. Uji Asumsi Klasik (Tanpa Multikolinearitas karena hanya 1 variabel X)
print("\nHASIL UJI ASUMSI KLASIK:")
shapiro_stat, shapiro_p = stats.shapiro(model_2v.resid)
print(f"- Normalitas (Shapiro-Wilk) P-Value: {shapiro_p:.4f} (>0.05 = Lolos)")

bp_test = sms.het_breuschpagan(model_2v.resid, model_2v.model.exog)
print(f"- Heteroskedastisitas (BP) P-Value   : {bp_test[1]:.4f} (>0.05 = Lolos)")

dw_value = sm.stats.stattools.durbin_watson(model_2v.resid)
print(f"- Autokorelasi (Durbin-Watson)       : {dw_value:.4f} (Mendekati 2 = Lolos)\n")

# 4. Visualisasi Grafik 2D Interaktif (Scatter Plot + Garis Regresi)
x_line = np.linspace(df['bm_ekonomi'].min(), df['bm_ekonomi'].max(), 100)
y_line = model_2v.params['const'] + (model_2v.params['bm_ekonomi'] * x_line)

fig_2d = go.Figure()

# Titik Data Aktual
fig_2d.add_trace(go.Scatter(
    x=df['bm_ekonomi'],
    y=df['PENGANGGURAN'],
    mode='markers',
    name='Data Aktual',
    marker=dict(size=10, color='red', line=dict(color='white', width=1))
))

# Garis Regresi Linear
fig_2d.add_trace(go.Scatter(
    x=x_line,
    y=y_line,
    mode='lines',
    name='Garis Regresi',
    line=dict(color='#00CC96', width=3)
))

# Layout Tampilan 2D
fig_2d.update_layout(
    title=f"Plot Regresi 2D: BM Ekonomi vs Pengangguran | R² = {model_2v.rsquared:.2f}",
    template='plotly_dark',
    xaxis_title='BM Ekonomi (X)',
    yaxis_title='Pengangguran (Y)',
    margin=dict(l=40, r=40, b=40, t=50)
)

fig_2d.show()

HASIL REGRESI 2 VARIABEL
                            OLS Regression Results                            
Dep. Variable:           PENGANGGURAN   R-squared:                       0.204
Model:                            OLS   Adj. R-squared:                  0.116
Method:                 Least Squares   F-statistic:                     2.313
Date:                Thu, 15 Oct 2026   Prob (F-statistic):              0.163
Time:                        01:42:34   Log-Likelihood:                -23.467
No. Observations:                  11   AIC:                             50.93
Df Residuals:                       9   BIC:                             51.73
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          3.6655      